In [1]:
import pandas as pd

# CSV-Datei einlesen
# Quelle: https://www.landesdatenbank.nrw.de/
df = pd.read_csv("../data/raw/2024/22811-05i.csv", sep=";", encoding="latin1", skiprows=4)

# Überblick:
print(df.head(3))
#print(df.info())
print(df.iloc[1])

  Sozialberichterstattung in der amtlichen Statistik  Unnamed: 1 Unnamed: 2  \
0                                                NaN         NaN        NaN   
1                                                NaN         NaN        NaN   
2                                                NaN         NaN        NaN   

                                          Unnamed: 3  \
0  Empfänger/innen von soz. Mindestsicherungsleis...   
1  Gesamtregelleistung (ALG II/Sozialgeld) nach d...   
2                                                NaN   

                                          Unnamed: 4  \
0  Empfänger/innen von soz. Mindestsicherungsleis...   
1                                          Insgesamt   
2                                                NaN   

                                          Unnamed: 5  \
0  Empfänger/innen von soz. Mindestsicherungsleis...   
1  Gesamtregelleistung (ALG II/Sozialgeld) nach d...   
2                                                NaN   

        

In [2]:
#relevante Spalten filtern
df = df[['Unnamed: 2','Unnamed: 3']]
df = df.dropna(axis=1, how="all")
df = df.dropna(axis=0, how="all")
print(df)

                          Unnamed: 2  \
0                                NaN   
1                                NaN   
3                                NaN   
4                Nordrhein-Westfalen   
5       Düsseldorf, Regierungsbezirk   
..                               ...   
433                  Schwerte, Stadt   
434                      Selm, Stadt   
435                      Unna, Stadt   
436                     Werne, Stadt   
437        Wohnort außerhalb von NRW   

                                            Unnamed: 3  
0    Empfänger/innen von soz. Mindestsicherungsleis...  
1    Gesamtregelleistung (ALG II/Sozialgeld) nach d...  
3                                               Anzahl  
4                                              1557269  
5                                               519326  
..                                                 ...  
433                                               3331  
434                                               1879  
435   

In [3]:
# Spalten umbenennen
df = df.rename(columns={"Unnamed: 2": "Name", "Unnamed: 3": "SGB II-Bezug"})

# nach Kreisen und kreisfreien Städten filtern und Strings anpassen
df = df[df["Name"].str.contains("Kreis|krfr. Stadt|Städteregion", case=False,na=False)]
df["Name"] = df["Name"].str.strip()

df["SGB II-Bezug"] = (
    df["SGB II-Bezug"]
    .astype(str)                
    .str.replace(",", ".", regex=False)
    .str.strip()
    .pipe(pd.to_numeric, errors="coerce")
)


print(df)

                                            Name  SGB II-Bezug
6                        Düsseldorf, krfr. Stadt         49723
7                          Duisburg, krfr. Stadt         71734
8                             Essen, krfr. Stadt         85382
9                           Krefeld, krfr. Stadt         25978
10                  Mönchengladbach, krfr. Stadt         33282
11              Mülheim an der Ruhr, krfr. Stadt         19445
12                       Oberhausen, krfr. Stadt         26251
13                        Remscheid, krfr. Stadt         10880
14                         Solingen, krfr. Stadt         14915
15                        Wuppertal, krfr. Stadt         45419
16                                  Kleve, Kreis         17414
33                               Mettmann, Kreis         39429
44                             Rhein-Kreis Neuss         30233
53                                Viersen, Kreis         17002
63                                  Wesel, Kreis       

In [4]:
# Zeile löschen und Name anpassen
df = df[df["Name"] != "Aachen, krfr. Stadt (ab 21.10.2009)"]
df = df[df["Name"] != "Essen, krfr. Stadt"]

# fehlende Werte anzeigen
print(df[df["SGB II-Bezug"].isna()])

Empty DataFrame
Columns: [Name, SGB II-Bezug]
Index: []


In [5]:
from ipynb.fs.defs.functions import clean_and_sort, validate_df
# Tabelle nach Kreis/Stadt sortieren
df = clean_and_sort(df, "SGB II-Bezug")
validate_df(
    df,
    not_null=["SGB II-Bezug"],
    non_negative=["SGB II-Bezug"],
    numeric=["SGB II-Bezug"],
    key_cols=["Name"],
)

DataFrame: alle Checks bestanden


[]

In [6]:
# Übersicht
df["SGB II-Bezug"].describe()

count        52.000000
mean      28305.519231
std       19890.687828
min        5690.000000
25%       16161.000000
50%       22717.000000
75%       32780.500000
max      108840.000000
Name: SGB II-Bezug, dtype: float64

In [7]:
# speichern
df.to_csv("../data/processed/sgb2_2024.csv", index=False)